# Text diversity selection

Using TF-IDF to convert each prompt into a vector based on its words and two-word phrases, thereby assigning more weight to distinctive terms.

For this method, I wanted to examine whether selecting prompts with more varied wording would produce effective training data for defense fine-tuning, using the same 256-pair selection size as the other methods.

- [Experiment settings](#Experiment-settings)
- [How text diversity selects the pairs](#How-text-diversity-selects-the-pairs)
- [Examples of the selected pairs](#Examples-of-the-selected-pairs)


## Experiment settings

I use the same Qwen2.5-7B experiment and FIPI candidate pool for this comparison, keeping the model and number of selected pairs consistent across all methods.

The function `configuration()` reads these settings into the dictionary variable `config`, while the variable `output` identifies the completed experiment used by the result-reading functions.


In [ ]:
# read the pairs selected for differences in wording

from pathlib import Path
import sys
import json
import pandas as pd

root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "scripts/curation/configs/config_qwen7b.json").exists())
sys.path.insert(0, str(root))

# use the same Qwen2.5-7B configuration across all selection methods
from scripts.curation.selection.extract_results import configuration, selection, validation
config, output = configuration()
pd.set_option("display.max_colwidth", None)

## How text diversity selects the pairs

TF-IDF stands for term frequency–inverse document frequency, meaning a term receives more weight when it is frequent in one prompt but less common across the dataset. Here the method represents each candidate's user instruction and injection content together.

I randomly sample one pair of prompts, then use cosine similarity to compare each remaining candidate with all already-selected pairs. For each candidate, I find its closest selected pair and choose the candidate with the lowest similarity to that pair, repeating until 256 pairs are selected.

The function `selection()` reads the completed selection into the table variable `selected`, while the column `selection_score` records the selection order using decreasing numbers.


In [ ]:
# compare the 256 pairs used to train this method's adapter
selected = selection("text_diversity")
# the counts show which injection methods appear in this training group
display(selected.groupby("attack_method").size().rename("selected pairs").to_frame())
display(selected[["pair_id", "selection_score", "witness_redirection"]].head(10))

## Examples of the selected pairs

The table below shows three selected pairs, each containing the original user instruction and both the safe and injection versions of its content. The source column `witness` gives the answer requested by the injection, so the inserted instruction can be compared with what it asks the model to produce.


In [ ]:
# keep the user instruction beside both prompt versions to inspect the inserted injection
display(selected[["user_instruction", "clean_data", "injected_data", "witness"]].head(3))